In [ ]:
!pip install torch numpy scipy scikit-learn matplotlib seaborn

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

import numpy as np
import scipy.io
import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

print("Libraries imported successfully!")

#Data Loading

In [ ]:
import os
import scipy.io
import numpy as np
import glob
import collections

DATA_FOLDER = 'C:/Users/tudor/EmotionRecognitionDemo/archive/seed_iv/eeg_feature_smooth'
FEATURE_PREFIX = 'de_LDS'

session1_labels = np.array([1,2,3,0,2,0,0,1,0,1,2,1,1,1,2,3,2,2,3,3,0,3,0,3])
session2_labels = np.array([2,1,3,0,0,2,0,2,3,3,2,3,2,0,1,1,2,1,0,3,0,1,3,1])
session3_labels = np.array([1,2,2,1,3,3,3,1,1,2,1,0,2,3,3,0,2,3,0,0,2,0,1,0])
label_dict = {'1': session1_labels, '2': session2_labels, '3': session3_labels}

print("Starting Data Loading")

all_data_buffer = []

file_pattern = DATA_FOLDER + "/**/*.mat"
all_files = glob.glob(file_pattern, recursive=True)

for file_path in all_files:
    session_num = os.path.basename(os.path.dirname(file_path))

    if session_num not in label_dict: continue

    try:
        data = scipy.io.loadmat(file_path)
        master_labels = label_dict[session_num]

        for i in range(24):
            trial_key = f"{FEATURE_PREFIX}{i+1}"
            trial_features = data[trial_key].transpose(1, 0, 2)

            label = master_labels[i]
            trial_labels = np.full(trial_features.shape[0], label)

            all_data_buffer.append({
                'session': session_num,
                'X': trial_features,
                'y': trial_labels
            })

    except Exception as e:
        print(f"Error loading {file_path}: {e}")

print(f"Successfully loaded {len(all_data_buffer)} trials from {len(all_files)} files.")


#Data Splitting

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

print("Splitting Data by Session")

X_train_list = []
y_train_list = []
X_test_list = []
y_test_list = []

for item in all_data_buffer:
    session_num = item['session']
    features = item['X']
    labels = item['y']

    if session_num in ['1', '2']:
        X_train_list.append(features)
        y_train_list.append(labels)

    elif session_num == '3':
        X_test_list.append(features)
        y_test_list.append(labels)

X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)
X_test = np.concatenate(X_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)

print(f"Training Data (Sessions 1+2): {X_train.shape}")
print(f"Testing Data (Session 3):     {X_test.shape}")

X_train_reshaped = np.expand_dims(X_train, axis=1)
X_test_reshaped = np.expand_dims(X_test, axis=1)

X_train_tensor = torch.tensor(X_train_reshaped, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_reshaped, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

BATCH_SIZE = 64

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("\nDataLoaders ready for Session-Based Training.")


#ResNet Block and Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Dropout Layer
        self.dropout = nn.Dropout(p=0.5)

        # Shortcut (skip connection)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        # Main path
        out = F.relu(self.bn1(self.conv1(x)))

        # Dropout Layer
        out = self.dropout(out)

        out = self.bn2(self.conv2(out))

        # Residual path (output = output + input)
        out += self.shortcut(x)

        out = F.relu(out)
        return out

class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=4): # 4 classes for SEED IV
        super(ResNet, self).__init__()
        self.in_channels = 64

        # MODIFICATION 1: First layer (1 channel input)
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        # ResNet layers
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        # Adaptive pooling (squashes to 1x1)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # Final Dropout
        self.dropout = nn.Dropout(p=0.5)

        # MODIFICATION 2: Final layer (4 emotion classes)
        self.linear = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_channels, out_channels, s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)

        out = self.dropout(out)

        out = self.linear(out)
        return out

def ResNet18_EEG():
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes=4)

print("ResNet model (with Dropout) defined.")



#Setup for Training

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

model = ResNet18_EEG()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model Setup")
print(f"Using device: {device}")

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=0.01)

print("Model, Criterion (Loss), and Optimizer (with Weight Decay) are ready.")


#Training and Testing Phases


In [ ]:
import time

NUM_EPOCHS = 25

print(f"--- Starting Training for {NUM_EPOCHS} Epochs ---")

history = {
    'train_loss': [],
    'train_acc': [],
    'test_loss': [],
    'test_acc': []
}

for epoch in range(NUM_EPOCHS):
    start_time = time.time()

    # Training phase
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        # FORWARD PASS: get model predictions
        outputs = model(inputs)

        loss = criterion(outputs, labels)

        # BACKWARD PASS: calculate gradients
        loss.backward()

        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct_train / total_train
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)

    # Testing phase
    model.eval()
    test_loss = 0.0
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    test_loss = test_loss / len(test_loader)
    test_acc = 100 * correct_test / total_test
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    end_time = time.time()

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Time: {end_time - start_time:.2f}s | "
          f"Train Loss: {train_loss:.4f} | "
          f"Train Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss:.4f} | "
          f"Test Acc: {test_acc:.2f}%")

print("--- Finished Training ---")


#Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from itertools import cycle

class_names = ['Neutral (0)', 'Sad (1)', 'Fear (2)', 'Happy (3)']

print("Predictions from Test Set")
model.eval()
softmax = nn.Softmax(dim=1)

with torch.no_grad():
    inputs, labels = next(iter(test_loader))
    inputs, labels = inputs.to(device), labels.to(device)

    outputs = model(inputs)
    probs = softmax(outputs)
    _, predicted = torch.max(outputs.data, 1)

    for i in range(10):
        true_label = class_names[labels[i].item()]
        predicted_label = class_names[predicted[i].item()]

        perc_neutral = probs[i][0].item() * 100
        perc_sad = probs[i][1].item() * 100
        perc_fear = probs[i][2].item() * 100
        perc_happy = probs[i][3].item() * 100

        print(f"\nSample {i+1}:")
        print(f"  TRUE Emotion:     {true_label}")
        print(f"  PREDICTED Emotion:  {predicted_label}")
        print(f"  Percentages:")
        print(f"    - Neutral: {perc_neutral:6.2f}%")
        print(f"    - Sad:     {perc_sad:6.2f}%")
        print(f"    - Fear:    {perc_fear:6.2f}%")
        print(f"    - Happy:   {perc_happy:6.2f}%")


all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        probs = softmax(outputs)
        _, predicted = torch.max(outputs.data, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

print("\n\n Full Evaluation Complete")
all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

print("\n Classification Report")
print(classification_report(all_labels, all_preds, target_names=class_names))

print("\n Confusion Matrix")
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (Subject-Independent)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print("\n ROC/AUC Analysis (One-vs-Rest)")
y_true_binarized = label_binarize(all_labels, classes=[0, 1, 2, 3])
n_classes = y_true_binarized.shape[1]

fpr, tpr, roc_auc = dict(), dict(), dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_binarized[:, i], all_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(10, 7))
colors = cycle(['aqua', 'darkorange', 'cornflowerblue', 'green'])
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC curve for class {0} (AUC = {1:0.2f})'
             ''.format(class_names[i], roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-Class ROC Analysis (One-vs-Rest)')
plt.legend(loc="lower right")
plt.show()

print("\nTraining History")
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['test_acc'], label='Test Accuracy')
plt.title('Model Accuracy vs. Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['test_loss'], label='Test Loss')
plt.title('Model Loss vs. Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()
